In [2]:
import os
import json

# CLAD 데이터셋 파악
# annotations 내 json 구조파악
with open('../data/SSLAD-2D/labeled/annotations/instance_train.json', mode = 'r') as f:
    json_data = json.load(f)
print(json_data.keys())
for i in range(5):
    for key in json_data.keys():
        print(key, len(json_data[key]))
        print(json_data[key][i])
    
# 파일 수 파악
for folder_list in os.listdir('../data/SSLAD-2D/labeled/'):
    print(folder_list, len(os.listdir(os.path.join('../data/SSLAD-2D/labeled', f'{folder_list}'))))

dict_keys(['categories', 'annotations', 'images'])
categories 6
{'supercategory': 'Pedestrian', 'id': 1, 'name': 'Pedestrian'}
annotations 41110
{'image_id': 1, 'category_id': 3, 'bbox': [65, 667, 174, 126], 'area': 21924, 'id': 1, 'truncated': -1, 'occluded': -1, 'iscrowd': 0}
images 5000
{'file_name': 'HT_TRAIN_000001_SH_000.jpg', 'file_name_old': '/home/ma-user/work/data/Haitian/Haitian_0005_13_VP/images/Vehicle_Person_Shanghai008896_00035.jpg', 'id': 1, 'height': 1080, 'width': 1920, 'city': 'Shanghai', 'location': 'Citystreet', 'period': 'Daytime', 'weather': 'Clear', 'date': '20181015', 'time': '112030'}
categories 6
{'supercategory': 'Cyclist', 'id': 2, 'name': 'Cyclist'}
annotations 41110
{'image_id': 1, 'category_id': 2, 'bbox': [418, 701, 30, 52], 'area': 1560, 'id': 2, 'truncated': -1, 'occluded': -1, 'iscrowd': 0}
images 5000
{'file_name': 'HT_TRAIN_000002_SH_000.jpg', 'file_name_old': '/home/ma-user/work/data/Haitian/Haitian_0005_10_VP/images/Vehicle_Person_Shanghai007986_

In [4]:
# Imagesets => train task에 따라 image index를 설정해놓은 값 => 무시
# 일단 PASCAL VOC처럼 JPEGImages를 저장해야함
# 다음과같이 일련번호 부여
# train_set => 000001.jpg ~ 005000.jpg
# val_set => 005001.jpg ~ 010000.jpg
# test_set => 010001.jpg ~ 020000.jpg
# 위 일련번호에 따라 파일을 이미지 파일 저장

from tqdm import tqdm

import shutil
count = 0
split_list = ['train','val', 'test']
for split in split_list:
    image_dir = os.path.join('../data/SSLAD-2D/labeled', split)
    file_name_list = os.listdir(image_dir)
    file_name_list.sort()
    for file_name in tqdm(file_name_list, desc = split):
        count += 1
        shutil.copyfile(os.path.join(image_dir, file_name), # 복사할 파일
                        'CLAD_PROB_FORMAT/data/OWOD/JPEGImages/'+f'{count:06}.jpg') # 복사될 위치

train:   0%|          | 0/5000 [00:00<?, ?it/s]

test: 100%|██████████| 10000/10000 [05:05<00:00, 32.78it/s]


In [6]:
# 이제 VOC Format에 맞게 Annotations 생성해야함
# VOC Format에만 존재하는 요소들은 다음과 같이 생략함
import json
import os
import xml.etree.ElementTree as ET
from xml.etree.ElementTree import Element, SubElement
from tqdm import tqdm

category_dict = {
    0 : 'Car',
    1 : 'Truck',
    2 : 'Tram',
    3 : 'Cyclist',
    4 : 'Tricycle',
    5 : 'Pedestrian'
}

def indent(elem, level=0):
    i = "\n" + level*"  "
    if len(elem):
        if not elem.text or not elem.text.strip():
            elem.text = i + "  "
        if not elem.tail or not elem.tail.strip():
            elem.tail = i
        for elem in elem:
            indent(elem, level+1)
        if not elem.tail or not elem.tail.strip():
            elem.tail = i
    else:
        if level and (not elem.tail or not elem.tail.strip()):
            elem.tail = i

def make_xml(output_dir, image_id, object_list):
    """
    VOC 형식의 XML을 생성하여 파일로 저장합니다.
    
    Parameters
    ----------
    output_dir : str
        XML 파일이 저장될 경로
    image_id : int
        현재 이미지 ID (파일명을 만드는데 사용)
    object_list : List[Dict]
        변환된 객체 정보 리스트
        예) [
              {
                'name': 'Car',
                'pose': 'Unknown',
                'truncated': 0,
                'difficult': 0,
                'bndbox': {
                    'xmin': 10,
                    'ymin': 20,
                    'xmax': 50,
                    'ymax': 60
                }
              },
              ...
            ]
    """
    # <annotation>
    annotation = Element('annotation')

    # <folder>VOC2007</folder>
    folder = SubElement(annotation, 'folder')
    folder.text = 'VOC2007'

    # <filename>000001.jpg</filename>
    filename_elem = SubElement(annotation, 'filename')
    # 예시로 image_id를 6자리로 맞춰서 jpg 파일명 생성
    filename_elem.text = f"{str(image_id).zfill(6)}.jpg"

    # <source> ... </source>
    source = SubElement(annotation, 'source')
    database = SubElement(source, 'database')
    database.text = 'The VOC2007 Database'
    annotation_source = SubElement(source, 'annotation')
    annotation_source.text = 'PASCAL VOC2007'
    image = SubElement(source, 'image')
    image.text = 'flickr'
    flickrid = SubElement(source, 'flickrid')
    flickrid.text = '341012865'

    # <owner> ... </owner>
    owner = SubElement(annotation, 'owner')
    owner_flickrid = SubElement(owner, 'flickrid')
    owner_flickrid.text = 'Fried Camels'
    owner_name = SubElement(owner, 'name')
    owner_name.text = 'Jinky the Fruit Bat'

    # <size> ... </size>
    size = SubElement(annotation, 'size')
    width = SubElement(size, 'width')
    width.text = '1920'   # 실제 이미지 크기에 맞춰 변경
    height = SubElement(size, 'height')
    height.text = '1080'   # 실제 이미지 크기에 맞춰 변경
    depth = SubElement(size, 'depth')
    depth.text = '3'      # 일반적인 RGB이미지라 가정

    # <segmented>0</segmented>
    segmented = SubElement(annotation, 'segmented')
    segmented.text = '0'

    # <object>...</object> * N
    for obj in object_list:
        object_elem = SubElement(annotation, 'object')

        name_elem = SubElement(object_elem, 'name')
        name_elem.text = obj['name']

        pose_elem = SubElement(object_elem, 'pose')
        # obj에 pose가 있다면 활용하고, 없으면 Unknown
        pose_elem.text = obj.get('pose', 'Unknown')

        truncated_elem = SubElement(object_elem, 'truncated')
        truncated_elem.text = str(obj.get('truncated', 0))

        difficult_elem = SubElement(object_elem, 'difficult')
        difficult_elem.text = str(obj.get('difficult', 0))

        bndbox = SubElement(object_elem, 'bndbox')
        xmin = SubElement(bndbox, 'xmin')
        xmin.text = str(obj['bndbox']['xmin'])

        ymin = SubElement(bndbox, 'ymin')
        ymin.text = str(obj['bndbox']['ymin'])

        xmax = SubElement(bndbox, 'xmax')
        xmax.text = str(obj['bndbox']['xmax'])

        ymax = SubElement(bndbox, 'ymax')
        ymax.text = str(obj['bndbox']['ymax'])

    # XML 트리를 파일로 저장
    tree = ET.ElementTree(annotation)
    os.makedirs(output_dir, exist_ok=True)
    xml_path = os.path.join(output_dir, f"{str(image_id).zfill(6)}.xml")
    tree.write(xml_path, encoding='utf-8', xml_declaration=False)

def create_voc_xml(annotations, output_dir, id_start, total_len):
    
    anno_idx_of_array = {}
    
    for image_id in range(1, 1 + total_len):
        anno_idx_of_array[image_id] = [i for i, annotation in enumerate(annotations) if annotation["image_id"] == image_id]
        
    for image_id in range(1, 1 + 10):
        print(f'image_id {image_id}, {anno_idx_of_array[image_id]}\n num of object : {len(anno_idx_of_array[image_id])}')
    
    for image_id in tqdm(range(1, 1 + total_len), desc = 'converting...', total = total_len):
        anno_idx_list = anno_idx_of_array[image_id]
        object_list = []
        for idx in anno_idx_list:
            object_list.append(annotations[idx])
        
        converted_object_list = []
        for object in object_list:
            # 기존순서가 x y w h 순서라 이에 맞게 수정 필요
            x ,y, w, h = object['bbox']
            xmin = int(x-w/2)
            xmax = int(x+w/2)
            ymin = int(y-h/2)
            ymax = int(y+h/2)
            converted_object_list.append({
                'name' : category_dict[object['category_id']],
                'pose' : 'Unknown',
                'truncated' : 0,
                'difficult' : 0,
                'bndbox' : {
                    'xmin' : xmin,
                    'ymin' : ymin,
                    'xmax' : xmax,
                    'ymax' : ymax
                }
            })
        make_xml(output_dir, image_id + id_start - 1, converted_object_list)

def json_to_voc_xml(json_file, output_dir):
    # Load JSON data
    with open(json_file, "r") as f:
        data = json.load(f)
    
    split = json_file.split('.')[-2].split('_')[-1]
    if split == 'train':
        id_start = 1
        total_len = 5000
    elif split == 'val':
        id_start = 5001
        total_len = 5000
    else:
        id_start = 10001
        total_len = 10000
    
    '''
    train : 5,000 장
    val : 5,000 장
    test : 10,000 장
    '''

    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Convert each annotation to VOC XML
    annotations = data['annotations']
    create_voc_xml(annotations, output_dir, id_start, total_len)    

for split in ['train', 'val', 'test']:
    # Example usage
    json_file = f"../data/SSLAD-2D/labeled/annotations/instance_{split}.json"  # Path to your JSON file
    output_dir = f"CLAD_PROB_FORMAT/data/OWOD/Annotations"         # Directory to save XML files
    json_to_voc_xml(json_file, output_dir)

image_id 1, [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]
 num of object : 25
image_id 2, [25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36]
 num of object : 12
image_id 3, [37, 38, 39, 40, 41, 42]
 num of object : 6
image_id 4, [43, 44, 45, 46, 47, 48, 49, 50, 51, 52]
 num of object : 10
image_id 5, [53, 54, 55, 56, 57, 58, 59, 60, 61]
 num of object : 9
image_id 6, [62, 63]
 num of object : 2
image_id 7, [64, 65, 66, 67]
 num of object : 4
image_id 8, [68, 69, 70]
 num of object : 3
image_id 9, [71, 72, 73, 74]
 num of object : 4
image_id 10, [75, 76, 77, 78, 79, 80]
 num of object : 6


converting...:   0%|          | 10/5000 [00:00<00:01, 3175.82it/s]


KeyError: 6